# Qwen3-VL OOF Resume on Colab A100 High-RAM

This notebook trains M1 OOF crop LoRA from scratch on Google Colab A100 + High RAM. It unzips local input archives to `/content/oof_inputs`, uses `oof_part1` + `oof_part2`, holds out a stratified 15% validation split by part and `region_type`, and trains with fixed `per_device_train_batch_size = 16` and `gradient_accumulation_steps = 16`.


In [ ]:
# Colab A100 setup cell.
# Runtime: Runtime -> Change runtime type -> GPU -> A100.
INSTALL_DEPS = True
MOUNT_GOOGLE_DRIVE = True

# Kaggle download is disabled for this local-input workflow.

import os
import subprocess
import sys
from pathlib import Path

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print("Google Drive mount skipped or failed:", repr(exc), flush=True)

if INSTALL_DEPS:
    commands = [
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade-strategy",
            "only-if-needed",
            "accelerate",
            "peft",
            "bitsandbytes",
            "trl",
            "qwen-vl-utils",
            "datasets",
            "hf_transfer",
            "pandas==2.2.2",
            "pillow<12",
        ],
        [sys.executable, "-m", "pip", "install", "-q", "-U", "git+https://github.com/huggingface/transformers.git"],
    ]
    for cmd in commands:
        print("Running:", " ".join(cmd), flush=True)
        subprocess.check_call(cmd)

try:
    subprocess.run(["nvidia-smi"], check=False)
except FileNotFoundError:
    print("nvidia-smi not found. Make sure Colab runtime has GPU enabled.", flush=True)


In [ ]:
import gc
import json
import math
import os
import random
import shutil
import subprocess
import time
import zipfile
from pathlib import Path

import torch
from datasets import Dataset

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    device_name = torch.cuda.get_device_name(0)
    print("GPU:", device_name)
else:
    raise RuntimeError("No CUDA GPU found. In Colab, choose Runtime -> Change runtime type -> GPU -> A100.")

# =========================
# EDIT ONLY THIS BLOCK
# =========================
# FOLD_ID=1 trains parts 1+2, FOLD_ID=2 trains parts 1+3, FOLD_ID=3 trains parts 2+3.
FOLD_ID = 1

# Upload/copy/unzip your inputs to this Colab local disk folder before training.
# Training will read from /content, not from Drive, so image loading is faster.
LOCAL_INPUT_ROOT = Path("/content/oof_inputs")

# Optional zip sources. Set this to the dataset zip you uploaded to Drive.
# oof_stage2b_crops.zip should unzip to /content/oof_inputs/oof_stage2b_crops.
LOCAL_INPUT_ZIP_PATHS = [
    Path("/content/drive/MyDrive/oof_stage2b_crops.zip"),
]
RESET_LOCAL_INPUT_ROOT_BEFORE_UNZIP = True
VALIDATION_FRACTION = 0.15

# Kaggle download is intentionally disabled in this notebook.

# Direct local path containing oof_part1, oof_part2, oof_part3.
CROP_DATA_ROOT = LOCAL_INPUT_ROOT / "oof_stage2b_crops"

# Leave empty when training a fresh LoRA from scratch.
START_LORA_DIR = ""

# Where the fresh LoRA and trainer checkpoints will be written.
OUTPUT_ROOT = Path("/content/drive/MyDrive/Handwritter to Data/OOF/OUTPUT/M1")

# Use the HF model id, or point this to a downloaded Qwen model folder containing config.json.
# If you upload/copy Qwen to local disk, set this to that folder.
# Example: BASE_MODEL_PATH = Path("/content/oof_inputs/Qwen3-VL-8B-Instruct")
BASE_MODEL_PATH = "Qwen/Qwen3-VL-8B-Instruct"
# =========================
# END EDIT BLOCK
# =========================

FOLD_TO_TRAIN_PARTS = {
    1: [1, 2],
    2: [1, 3],
    3: [2, 3],
}
if FOLD_ID not in FOLD_TO_TRAIN_PARTS:
    raise ValueError(f"FOLD_ID must be one of {sorted(FOLD_TO_TRAIN_PARTS)}, got {FOLD_ID}")
TRAIN_PARTS = FOLD_TO_TRAIN_PARTS[FOLD_ID]
HELD_OUT_PART = sorted(set([1, 2, 3]) - set(TRAIN_PARTS))[0]
RUN_NAME = f"model{FOLD_ID}_parts{'_'.join(str(p) for p in TRAIN_PARTS)}"

# Train from scratch: create a new LoRA adapter on top of the base Qwen model.
RESUME_MODE = "from_scratch"
RESUME_CHECKPOINT_DIR = ""
PREVIOUS_EPOCHS_DONE = 0
NUM_TRAIN_EPOCHS = 3


def log_local_input_tree(root, max_depth=3, max_items=120):
    root = Path(root).expanduser()
    if not root.exists():
        print("Local input root does not exist yet:", root, flush=True)
        return

    print("Local input root:", root, flush=True)
    shown = 0
    for path in sorted(root.rglob("*")):
        rel = path.relative_to(root)
        depth = len(rel.parts)
        if depth > max_depth:
            continue
        prefix = "  " * depth
        suffix = "/" if path.is_dir() else ""
        print(f"{prefix}{rel.name}{suffix}", flush=True)
        shown += 1
        if shown >= max_items:
            print(f"  ... stopped after {max_items} items", flush=True)
            break


def normalize_zip_paths(zip_paths):
    if zip_paths is None:
        return []
    if isinstance(zip_paths, (str, Path)):
        zip_paths = [zip_paths]

    normalized = []
    for item in zip_paths:
        item = str(item or "").strip()
        if item:
            normalized.append(Path(item).expanduser())
    return normalized


def prepare_local_inputs_from_zips(zip_paths, extract_root, reset=True):
    zip_paths = normalize_zip_paths(zip_paths)
    extract_root = Path(extract_root).expanduser()
    if not zip_paths:
        print("No input zips configured. Using existing local folders under:", extract_root, flush=True)
        log_local_input_tree(extract_root)
        return extract_root

    for zip_path in zip_paths:
        if not zip_path.exists():
            raise FileNotFoundError(f"LOCAL_INPUT_ZIP_PATHS item does not exist: {zip_path}")
        if not zip_path.is_file():
            raise FileNotFoundError(f"LOCAL_INPUT_ZIP_PATHS item is not a file: {zip_path}")

    if reset and extract_root.exists():
        print("Removing old local input root:", extract_root, flush=True)
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True, exist_ok=True)

    unzip_bin = shutil.which("unzip")
    for zip_path in zip_paths:
        print("Unzipping input zip:", zip_path, flush=True)
        print("Extract to:", extract_root, flush=True)
        if unzip_bin:
            subprocess.check_call([unzip_bin, "-q", str(zip_path), "-d", str(extract_root)])
        else:
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall(extract_root)
    print("All input zips unzipped.", flush=True)
    log_local_input_tree(extract_root)
    return extract_root


print("Kaggle download disabled. Using direct local paths under /content.", flush=True)
LOCAL_INPUT_ROOT = prepare_local_inputs_from_zips(
    LOCAL_INPUT_ZIP_PATHS,
    LOCAL_INPUT_ROOT,
    reset=RESET_LOCAL_INPUT_ROOT_BEFORE_UNZIP,
)

def resolve_dir(path, name, must_exist=True):
    path = Path(path).expanduser()
    if must_exist:
        if not path.exists():
            raise FileNotFoundError(f"{name} does not exist: {path}")
        if not path.is_dir():
            raise NotADirectoryError(f"{name} is not a directory: {path}")
    return path


def require_file(path, name):
    path = Path(path).expanduser()
    if not path.exists():
        raise FileNotFoundError(f"{name} does not exist: {path}")
    if not path.is_file():
        raise FileNotFoundError(f"{name} is not a file: {path}")
    return path


def validate_direct_crop_root(root):
    root = resolve_dir(root, "CROP_DATA_ROOT")
    for part in [1, 2, 3]:
        resolve_dir(root / f"oof_part{part}", f"CROP_DATA_ROOT/oof_part{part}")
    for part in TRAIN_PARTS:
        require_file(
            root / f"oof_part{part}" / "stage2b_oof_crop_samples.jsonl",
            f"stage2b_oof_crop_samples.jsonl for oof_part{part}",
        )
    return root


def validate_direct_lora_dir(root):
    root = resolve_dir(root, "START_LORA_DIR")
    require_file(root / "adapter_config.json", "LoRA adapter_config.json")
    safetensors_path = root / "adapter_model.safetensors"
    bin_path = root / "adapter_model.bin"
    if not safetensors_path.exists() and not bin_path.exists():
        raise FileNotFoundError(f"No adapter_model.safetensors or adapter_model.bin found in START_LORA_DIR={root}")
    return root


def resolve_base_model_path(path_or_id):
    item = str(path_or_id)
    if item.startswith("/") or item.startswith("~"):
        root = resolve_dir(Path(item).expanduser(), "BASE_MODEL_PATH")
        require_file(root / "config.json", "Qwen base model config.json")
        print("Using local Qwen base model:", root, flush=True)
        return root
    print("Using Hugging Face Qwen base model id:", item, flush=True)
    return item


CROP_DATA_ROOT = validate_direct_crop_root(CROP_DATA_ROOT)
START_LORA_DIR = validate_direct_lora_dir(START_LORA_DIR) if RESUME_MODE == "lora_adapter" else None
BASE_MODEL_PATH = resolve_base_model_path(BASE_MODEL_PATH)
OUTPUT_ROOT = resolve_dir(OUTPUT_ROOT, "OUTPUT_ROOT", must_exist=False)

OUTPUT_DIR = OUTPUT_ROOT
MODEL_OUTPUT_DIR = OUTPUT_DIR / RUN_NAME
TRAINER_OUTPUT_DIR = MODEL_OUTPUT_DIR / "trainer_checkpoints"
FINAL_DIR = MODEL_OUTPUT_DIR / f"qwen3vl_rukopys_oof_{RUN_NAME}_lora_final"
for path in [OUTPUT_DIR, MODEL_OUTPUT_DIR, TRAINER_OUTPUT_DIR, FINAL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# Fixed A100 High-RAM setting requested: per-device batch 16 x grad_accum 16.
GPU_TOTAL_GIB = torch.cuda.get_device_properties(0).total_memory / 1024**3
MAX_SPEED_MODE = False
AUTO_TUNE_BATCH_SIZE = False
BATCH_SIZE_CANDIDATES = [16]
TARGET_EFFECTIVE_BATCH = 256
PER_DEVICE_BATCH = 16
GRAD_ACCUM = 16
DATALOADER_NUM_WORKERS = 4
MODEL_DEVICE_MAP = "auto"
MAX_SEQ_LENGTH = 3072
MAX_PIXELS_CROP = 320_000

LEARNING_RATE = 1e-4
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

LOGGING_STEPS = 1
SAVE_STEPS = 60
SAVE_TOTAL_LIMIT = 2
RESUME_TRAINING = False

CUDA_MAJOR = torch.cuda.get_device_capability(0)[0]
USE_BF16 = CUDA_MAJOR >= 8
TORCH_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

print("Config OK")
print("Input mode: local Colab disk (/content); Kaggle download disabled")
print("FOLD_ID:", FOLD_ID)
print("Training parts:", TRAIN_PARTS, "held out for later:", HELD_OUT_PART)
print("Run name:", RUN_NAME)
print("Resume mode:", RESUME_MODE)
print("Epochs to run:", NUM_TRAIN_EPOCHS)
print("Validation fraction:", VALIDATION_FRACTION)
print("GPU total memory:", f"{GPU_TOTAL_GIB:.1f} GiB")
print("Max speed mode:", MAX_SPEED_MODE)
print("Autotune batch size:", AUTO_TUNE_BATCH_SIZE)
print("Batch candidates:", BATCH_SIZE_CANDIDATES)
print("Initial per-device batch / grad accum:", PER_DEVICE_BATCH, "/", GRAD_ACCUM)
print("Torch dtype:", TORCH_DTYPE)
print("Crop data root:", CROP_DATA_ROOT)
print("Start LoRA dir:", START_LORA_DIR)
print("Base model path/id:", BASE_MODEL_PATH)
print("Trainer output:", TRAINER_OUTPUT_DIR)
print("Final LoRA output:", FINAL_DIR)


In [ ]:
def find_model_id():
    item = str(BASE_MODEL_PATH)
    if item.startswith("/") and Path(item).exists():
        return item
    if not item.startswith("/"):
        return item
    raise FileNotFoundError(f"No Qwen3-VL model found at BASE_MODEL_PATH={item}")


def find_start_lora_dir():
    if RESUME_MODE != "lora_adapter":
        return None
    p = Path(START_LORA_DIR)
    if not (p / "adapter_config.json").exists():
        raise FileNotFoundError(f"No adapter_config.json found in START_LORA_DIR={p}")
    if not (p / "adapter_model.safetensors").exists() and not (p / "adapter_model.bin").exists():
        raise FileNotFoundError(f"No adapter_model.safetensors or adapter_model.bin found in START_LORA_DIR={p}")
    return p


def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def part_dir(root, part):
    return Path(root) / f"oof_part{part}"


def part_manifest(root, part):
    return part_dir(root, part) / "stage2b_oof_crop_samples.jsonl"


def root_has_parts(root, parts):
    root = Path(root)
    return all(part_manifest(root, part).exists() for part in parts)


def available_parts(root):
    root = Path(root)
    found = []
    for part in [1, 2, 3]:
        if part_manifest(root, part).exists():
            found.append(part)
    return found


def discover_crop_data_root(required_parts):
    root = Path(CROP_DATA_ROOT)
    if not root.exists():
        raise FileNotFoundError(f"CROP_DATA_ROOT does not exist: {root}")
    if not root.is_dir():
        raise NotADirectoryError(f"CROP_DATA_ROOT is not a directory: {root}")
    missing = []
    for part in [1, 2, 3]:
        if not part_dir(root, part).exists():
            missing.append(str(part_dir(root, part)))
    for part in required_parts:
        if not part_manifest(root, part).exists():
            missing.append(str(part_manifest(root, part)))
    if missing:
        raise FileNotFoundError("CROP_DATA_ROOT is missing required paths:\n" + "\n".join(missing))
    return root


def resolve_cached_image_path(root, part, row):
    part_root = part_dir(root, part)
    raw = Path(row.get("image_path") or row.get("relative_image_path") or "")
    if raw.is_absolute() and raw.exists():
        return str(raw)

    candidates = [part_root / raw, Path(root) / raw]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)

    raise FileNotFoundError(f"Cached crop not found for part {part}: {raw} under {part_root}")


def load_part_samples(root, part):
    manifest_path = part_manifest(root, part)
    rows = read_jsonl(manifest_path)
    out = []
    for row in rows:
        row = dict(row)
        row["image_path"] = resolve_cached_image_path(root, part, row)
        row["fold_part"] = part
        out.append(row)
    return out


model_id = find_model_id()
start_lora_dir = find_start_lora_dir()
crop_data_root = discover_crop_data_root(TRAIN_PARTS)

part_samples = {part: load_part_samples(crop_data_root, part) for part in TRAIN_PARTS}
samples = []
for part in TRAIN_PARTS:
    samples.extend(part_samples[part])
random.Random(SEED).shuffle(samples)

if not samples:
    raise RuntimeError(f"No training samples found for FOLD_ID={FOLD_ID}, parts={TRAIN_PARTS}")


def validate_training_samples(rows):
    problems = []
    by_part = {}
    by_region_type = {}
    for idx, row in enumerate(rows):
        part = row.get("fold_part", "unknown")
        rtype = str(row.get("region_type", "unknown"))
        by_part[part] = by_part.get(part, 0) + 1
        by_region_type[rtype] = by_region_type.get(rtype, 0) + 1

        image_path = row.get("image_path")
        prompt = row.get("prompt")
        answer = row.get("answer")
        if not image_path:
            problems.append(f"row {idx}: missing image_path")
        elif not Path(image_path).exists():
            problems.append(f"row {idx}: image does not exist: {image_path}")
        if prompt is None or not str(prompt).strip():
            problems.append(f"row {idx}: missing or empty prompt")
        if answer is None or not str(answer).strip():
            problems.append(f"row {idx}: missing or empty answer")

    print("Dataset validation summary:")
    print("  rows:", len(rows))
    print("  by part:", dict(sorted(by_part.items())))
    print("  by region_type:", dict(sorted(by_region_type.items())))
    if problems:
        preview = "\n".join(problems[:20])
        raise RuntimeError(f"Dataset validation failed with {len(problems)} problems. First problems:\n{preview}")
    print("Dataset validation OK: image_path, prompt, and answer are present.")


def count_samples_by_part(rows):
    counts = {}
    for row in rows:
        part = row.get("fold_part", "unknown")
        counts[part] = counts.get(part, 0) + 1
    return dict(sorted(counts.items()))


def count_samples_by_region_type(rows):
    counts = {}
    for row in rows:
        region_type = str(row.get("region_type", "unknown"))
        counts[region_type] = counts.get(region_type, 0) + 1
    return dict(sorted(counts.items()))


def count_samples_by_part_and_type(rows):
    counts = {}
    for row in rows:
        key = f"part{row.get('fold_part', 'unknown')}::{row.get('region_type', 'unknown')}"
        counts[key] = counts.get(key, 0) + 1
    return dict(sorted(counts.items()))


def split_train_validation_samples(rows, val_fraction, seed):
    val_fraction = float(val_fraction or 0.0)
    if val_fraction <= 0:
        return list(rows), []
    if val_fraction >= 0.5:
        raise ValueError("VALIDATION_FRACTION should be below 0.5 for this training setup")

    rng = random.Random(seed)
    by_part_and_type = {}
    for row in rows:
        key = (row.get("fold_part", "unknown"), str(row.get("region_type", "unknown")))
        by_part_and_type.setdefault(key, []).append(row)

    train_rows = []
    val_rows = []
    for key, group_rows in sorted(by_part_and_type.items(), key=lambda item: (str(item[0][0]), item[0][1])):
        group_rows = list(group_rows)
        rng.shuffle(group_rows)
        if len(group_rows) <= 1:
            train_rows.extend(group_rows)
            continue
        val_count = max(1, int(round(len(group_rows) * val_fraction)))
        val_count = min(val_count, len(group_rows) - 1)
        val_rows.extend(group_rows[:val_count])
        train_rows.extend(group_rows[val_count:])

    rng.shuffle(train_rows)
    rng.shuffle(val_rows)
    return train_rows, val_rows


validate_training_samples(samples)
train_samples, val_samples = split_train_validation_samples(samples, VALIDATION_FRACTION, SEED)
if not train_samples:
    raise RuntimeError("No training samples remain after validation split")
stage2_train_ds = Dataset.from_list(train_samples)
stage2_eval_ds = Dataset.from_list(val_samples) if val_samples else None
stage2_ds = stage2_train_ds

prompt_configs = {}
for part in TRAIN_PARTS:
    path = part_dir(crop_data_root, part) / "prompt_config.json"
    if path.exists():
        prompt_configs[f"part{part}"] = json.loads(path.read_text(encoding="utf-8"))

print("Base model:", model_id)
if start_lora_dir:
    print("Starting LoRA:", start_lora_dir)
print("Crop data root:", crop_data_root)
print("Available crop parts:", available_parts(crop_data_root))
print("Samples total before split:", len(samples))
print("Validation fraction:", VALIDATION_FRACTION)
print("Train samples total:", len(train_samples), count_samples_by_part(train_samples))
print("Validation samples total:", len(val_samples), count_samples_by_part(val_samples))
print("Train by region_type:", count_samples_by_region_type(train_samples))
print("Validation by region_type:", count_samples_by_region_type(val_samples))
print("Validation by part/type:", count_samples_by_part_and_type(val_samples))
for part in TRAIN_PARTS:
    print(f"  part{part} samples:", len(part_samples[part]))
print("Example keys:", sorted(samples[0].keys()))
print("Example image:", samples[0]["image_path"])
print("Example answer:", str(samples[0].get("answer", ""))[:200])


In [ ]:
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from qwen_vl_utils import process_vision_info
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig, TrainerCallback
from trl import SFTConfig, SFTTrainer


def force_model_dtype_config(model, torch_dtype):
    model.config.torch_dtype = torch_dtype
    dtype_name = "bfloat16" if torch_dtype is torch.bfloat16 else "float16"
    for attr in ("text_config", "vision_config"):
        cfg = getattr(model.config, attr, None)
        if cfg is not None:
            cfg.torch_dtype = torch_dtype
            if hasattr(cfg, "dtype"):
                cfg.dtype = dtype_name


def make_lora_config():
    kwargs = dict(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
    )
    try:
        return LoraConfig(**kwargs, use_rslora=True)
    except TypeError:
        return LoraConfig(**kwargs)


def build_max_memory(reserve_gib=6):
    max_memory = {}
    for idx in range(torch.cuda.device_count()):
        total_gib = torch.cuda.get_device_properties(idx).total_memory // 1024**3
        max_memory[idx] = f"{max(8, total_gib - reserve_gib)}GiB"
    return max_memory or None


processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=TORCH_DTYPE,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

base_model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    device_map=MODEL_DEVICE_MAP,
    max_memory=build_max_memory(),
    quantization_config=quantization_config,
    dtype=TORCH_DTYPE,
    trust_remote_code=True,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)
force_model_dtype_config(base_model, TORCH_DTYPE)
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)
try:
    base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
except TypeError:
    base_model.gradient_checkpointing_enable()

if RESUME_MODE == "lora_adapter":
    model = PeftModel.from_pretrained(base_model, str(start_lora_dir), is_trainable=True)
else:
    print("Creating a fresh LoRA adapter from scratch.", flush=True)
    model = get_peft_model(base_model, make_lora_config())

for _, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)
model.print_trainable_parameters()


In [ ]:
def build_messages(sample):
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": sample["image_path"], "max_pixels": MAX_PIXELS_CROP},
                {"type": "text", "text": sample["prompt"]},
            ],
        },
        {"role": "assistant", "content": [{"type": "text", "text": sample["answer"]}]},
    ]


def encode_marker(tokenizer):
    try:
        return tokenizer.encode("<|im_start|>assistant\n", allowed_special="all", add_special_tokens=False)
    except TypeError:
        return tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)


ASSISTANT_MARKER = encode_marker(processor.tokenizer)


def data_collator(examples):
    messages_list = [build_messages(ex) for ex in examples]
    texts = [
        processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=False)
        for msg in messages_list
    ]
    image_inputs, video_inputs = process_vision_info(messages_list)
    batch = processor(
        text=texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        return_tensors="pt",
    )

    labels = batch["input_ids"].clone()
    pad_id = processor.tokenizer.pad_token_id
    if pad_id is not None:
        labels[labels == pad_id] = -100

    eos_id = processor.tokenizer.eos_token_id
    for i in range(labels.shape[0]):
        ids = batch["input_ids"][i].tolist()
        start = -1
        for j in range(0, len(ids) - len(ASSISTANT_MARKER) + 1):
            if ids[j:j + len(ASSISTANT_MARKER)] == ASSISTANT_MARKER:
                start = j + len(ASSISTANT_MARKER)
                break
        actual_len = int(batch["attention_mask"][i].sum().item())
        truncated = actual_len >= MAX_SEQ_LENGTH and (eos_id is None or ids[actual_len - 1] != eos_id)
        if start >= 0 and not truncated:
            labels[i, :start] = -100
        else:
            labels[i, :] = -100

    batch["labels"] = labels
    for key, value in list(batch.items()):
        if isinstance(value, torch.Tensor) and value.dtype == torch.float32:
            batch[key] = value.to(TORCH_DTYPE)
    return batch


In [ ]:

import inspect

from IPython.display import clear_output
from tqdm.auto import tqdm
from transformers import PrinterCallback, ProgressCallback


class NotebookProgressCallback(TrainerCallback):
    def __init__(self, name):
        self.name = name
        self.start_time = None
        self.start_step = 0
        self.pbar = None
        self.last_step = 0

    def _format_value(self, key, value):
        try:
            value = float(value)
        except Exception:
            return value
        if key == "learning_rate":
            return f"{value:.2e}"
        if key == "num_tokens":
            return f"{value:.0f}"
        if key in {"loss", "eval_loss", "mean_token_accuracy", "entropy"}:
            return f"{value:.4f}"
        if key == "grad_norm":
            return f"{value:.2f}"
        return f"{value:.4g}"

    def _postfix(self, logs):
        logs = logs or {}
        key_map = [
            ("loss", "loss"),
            ("eval_loss", "val_loss"),
            ("learning_rate", "lr"),
            ("grad_norm", "grad"),
            ("mean_token_accuracy", "acc"),
            ("entropy", "entropy"),
            ("num_tokens", "tokens"),
        ]
        return {label: self._format_value(key, logs[key]) for key, label in key_map if key in logs}

    def _epoch_text(self, state, args):
        total_epochs = max(1, int(math.ceil(float(args.num_train_epochs))))
        epoch_value = float(state.epoch or 0.0)
        current = min(max(1, math.ceil(epoch_value if epoch_value > 0 else 1)), total_epochs)
        return f"Epoch {current}/{total_epochs}"

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        self.start_step = int(state.global_step or 0)
        self.last_step = self.start_step
        clear_output(wait=True)
        total = int(state.max_steps or 0)
        self.pbar = tqdm(
            total=total,
            initial=self.start_step,
            desc=f"{self._epoch_text(state, args)} {self.name}",
            dynamic_ncols=True,
            leave=True,
        )

    def on_log(self, args, state, control, logs=None, **kwargs):
        if self.pbar is None:
            return
        step = int(state.global_step or 0)
        if step > self.last_step:
            self.pbar.update(step - self.last_step)
            self.last_step = step
        self.pbar.set_description(f"{self._epoch_text(state, args)} {self.name}")
        postfix = self._postfix(logs)
        if postfix:
            self.pbar.set_postfix(postfix, refresh=True)

    def on_step_end(self, args, state, control, **kwargs):
        if self.pbar is None:
            return
        step = int(state.global_step or 0)
        if step > self.last_step:
            self.pbar.update(step - self.last_step)
            self.last_step = step
        self.pbar.set_description(f"{self._epoch_text(state, args)} {self.name}")

    def on_epoch_end(self, args, state, control, **kwargs):
        control.should_save = True
        if self.pbar is not None:
            self.pbar.set_description(f"{self._epoch_text(state, args)} {self.name}")
        return control

    def on_save(self, args, state, control, **kwargs):
        if self.pbar is not None:
            self.pbar.write(f"saved checkpoint-{state.global_step}")

    def on_train_end(self, args, state, control, **kwargs):
        if self.pbar is not None:
            step = int(state.global_step or 0)
            if step > self.last_step:
                self.pbar.update(step - self.last_step)
            self.pbar.close()
            self.pbar = None


class OOMRecoverySFTTrainer(SFTTrainer):
    def training_step(self, model, inputs, num_items_in_batch=None):
        try:
            try:
                loss = super().training_step(model, inputs, num_items_in_batch=num_items_in_batch)
            except TypeError:
                loss = super().training_step(model, inputs)
            if self.args.device != loss.device:
                loss = loss.to(self.args.device)
            return loss
        except torch.cuda.OutOfMemoryError:
            print("OOM: skipping one batch after clearing cache.", flush=True)
            for p in model.parameters():
                p.grad = None
            torch.cuda.empty_cache()
            gc.collect()
            return torch.tensor(0.0, device=self.args.device)


def find_latest_checkpoint(root):
    root = Path(root)
    if not root.exists():
        return None
    checkpoints = []
    for path in root.glob("checkpoint-*"):
        try:
            step = int(path.name.rsplit("-", 1)[-1])
        except ValueError:
            continue
        if (path / "trainer_state.json").exists():
            checkpoints.append((step, path))
    if not checkpoints:
        return None
    return str(max(checkpoints, key=lambda item: item[0])[1])


def resolve_resume_checkpoint(output_dir):
    if not RESUME_TRAINING:
        return None
    if RESUME_CHECKPOINT_DIR:
        resume_root = Path(RESUME_CHECKPOINT_DIR)
        if not resume_root.exists():
            raise FileNotFoundError(f"RESUME_CHECKPOINT_DIR does not exist: {resume_root}")
        if (resume_root / "trainer_state.json").exists():
            return str(resume_root)
        latest = find_latest_checkpoint(resume_root)
        if latest:
            return latest
        raise FileNotFoundError(f"No checkpoint-* with trainer_state.json found under {resume_root}")
    return find_latest_checkpoint(output_dir)


def get_input_device(model):
    for param in model.parameters():
        if param.device.type == "cuda":
            return param.device
    return torch.device("cuda:0")


def clear_cuda_memory():
    for param in model.parameters():
        param.grad = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def select_probe_examples(rows, batch_size):
    ranked = sorted(
        rows,
        key=lambda row: len(str(row.get("prompt", ""))) + len(str(row.get("answer", ""))),
        reverse=True,
    )
    if len(ranked) >= batch_size:
        return ranked[:batch_size]
    repeats = math.ceil(batch_size / max(1, len(ranked)))
    return (ranked * repeats)[:batch_size]


def batch_to_device(batch, device):
    moved = {}
    for key, value in batch.items():
        if isinstance(value, torch.Tensor):
            moved[key] = value.to(device)
        else:
            moved[key] = value
    return moved


def probe_batch_size(batch_size):
    clear_cuda_memory()
    probe_examples = select_probe_examples(samples, batch_size)
    input_device = get_input_device(model)
    model.train()
    try:
        batch = batch_to_device(data_collator(probe_examples), input_device)
        with torch.amp.autocast("cuda", dtype=TORCH_DTYPE):
            outputs = model(**batch)
            loss = outputs.loss
        loss.backward()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            peak_gib = torch.cuda.max_memory_allocated() / 1024**3
        else:
            peak_gib = 0.0
        print(f"Batch probe OK: batch={batch_size}, peak_vram={peak_gib:.1f}GB", flush=True)
        return True, peak_gib
    except torch.cuda.OutOfMemoryError:
        print(f"Batch probe OOM: batch={batch_size}", flush=True)
        return False, None
    except RuntimeError as exc:
        if "out of memory" in str(exc).lower():
            print(f"Batch probe OOM: batch={batch_size}", flush=True)
            return False, None
        raise
    finally:
        clear_cuda_memory()


def tune_batch_size_for_a100():
    global PER_DEVICE_BATCH, GRAD_ACCUM
    if not AUTO_TUNE_BATCH_SIZE:
        print(f"Batch autotune disabled: batch={PER_DEVICE_BATCH}, grad_accum={GRAD_ACCUM}", flush=True)
        return

    candidates = sorted(set(int(x) for x in BATCH_SIZE_CANDIDATES if int(x) > 0), reverse=True)
    if not candidates:
        raise ValueError("BATCH_SIZE_CANDIDATES must contain at least one positive integer")

    print("Autotuning per-device batch candidates:", candidates, flush=True)
    for candidate in candidates:
        ok, _ = probe_batch_size(candidate)
        if ok:
            PER_DEVICE_BATCH = candidate
            GRAD_ACCUM = 1 if MAX_SPEED_MODE else max(1, math.ceil(TARGET_EFFECTIVE_BATCH / PER_DEVICE_BATCH))
            print(
                f"Selected batch={PER_DEVICE_BATCH}, grad_accum={GRAD_ACCUM}, "
                f"effective_batch={PER_DEVICE_BATCH * GRAD_ACCUM}",
                flush=True,
            )
            return

    raise RuntimeError("No batch size candidate fit in GPU memory, including batch=1.")


tune_batch_size_for_a100()

def sft_config_supports_arg(name):
    try:
        return name in inspect.signature(SFTConfig.__init__).parameters
    except Exception:
        return False


def filter_sft_config_kwargs(kwargs):
    try:
        params = inspect.signature(SFTConfig.__init__).parameters
    except Exception:
        return dict(kwargs)

    if any(param.kind == inspect.Parameter.VAR_KEYWORD for param in params.values()):
        return dict(kwargs)

    supported = set(params)
    filtered = {key: value for key, value in kwargs.items() if key in supported}
    skipped = sorted(set(kwargs) - set(filtered))
    if skipped:
        print("SFTConfig skipped unsupported args:", skipped, flush=True)
    return filtered


config_kwargs = dict(
    output_dir=str(TRAINER_OUTPUT_DIR),
    overwrite_output_dir=True,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    fp16=not USE_BF16,
    bf16=USE_BF16,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_grad_norm=0.3,
    logging_strategy="steps",
    logging_first_step=True,
    logging_steps=LOGGING_STEPS,
    disable_tqdm=True,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=True,
    dataloader_num_workers=DATALOADER_NUM_WORKERS,
    dataloader_pin_memory=False,
    dataloader_persistent_workers=DATALOADER_NUM_WORKERS > 0,
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
)

eval_strategy_name = None
if stage2_eval_ds is not None:
    if sft_config_supports_arg("eval_strategy"):
        config_kwargs["eval_strategy"] = "epoch"
        eval_strategy_name = "eval_strategy"
    elif sft_config_supports_arg("evaluation_strategy"):
        config_kwargs["evaluation_strategy"] = "epoch"
        eval_strategy_name = "evaluation_strategy"
    if sft_config_supports_arg("do_eval"):
        config_kwargs["do_eval"] = True
    print("Validation enabled: eval loss will be computed once per epoch.", flush=True)
else:
    print("Validation disabled: no validation samples were created.", flush=True)

training_args = SFTConfig(**filter_sft_config_kwargs(config_kwargs))

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=stage2_train_ds,
    data_collator=data_collator,
    callbacks=[NotebookProgressCallback(RUN_NAME)],
)
if stage2_eval_ds is not None:
    trainer_kwargs["eval_dataset"] = stage2_eval_ds

trainer = OOMRecoverySFTTrainer(**trainer_kwargs)
trainer.remove_callback(PrinterCallback)
trainer.remove_callback(ProgressCallback)

resume_checkpoint = None
print("Starting a fresh LoRA run from the base model; no checkpoint or adapter is loaded.", flush=True)
trainer.train()

trainer.model.save_pretrained(FINAL_DIR)
processor.save_pretrained(FINAL_DIR)

train_config = {
    "stage": "oof_3model_crop_lora_from_scratch_colab_a100_val_split",
    "fold_id": FOLD_ID,
    "train_parts": TRAIN_PARTS,
    "held_out_part_for_later": HELD_OUT_PART,
    "run_name": RUN_NAME,
    "base_model": str(model_id),
    "start_lora": str(start_lora_dir) if start_lora_dir else None,
    "torch_dtype": "bfloat16" if USE_BF16 else "float16",
    "crop_data_root": str(crop_data_root),
    "input_lora_dir": str(start_lora_dir) if start_lora_dir else None,
    "output_root": str(OUTPUT_ROOT),
    "all_samples_total": len(samples),
    "loaded_samples_by_part": {f"part{part}": len(part_samples[part]) for part in TRAIN_PARTS},
    "validation_fraction": VALIDATION_FRACTION,
    "train_samples_total": len(train_samples),
    "train_samples_by_part": count_samples_by_part(train_samples),
    "train_samples_by_region_type": count_samples_by_region_type(train_samples),
    "train_samples_by_part_and_type": count_samples_by_part_and_type(train_samples),
    "validation_samples_total": len(val_samples),
    "validation_samples_by_part": count_samples_by_part(val_samples),
    "validation_samples_by_region_type": count_samples_by_region_type(val_samples),
    "validation_samples_by_part_and_type": count_samples_by_part_and_type(val_samples),
    "eval_strategy": "epoch" if stage2_eval_ds is not None else None,
    "resume_mode": RESUME_MODE,
    "previous_epochs_done": PREVIOUS_EPOCHS_DONE,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "max_speed_mode": MAX_SPEED_MODE,
    "auto_tune_batch_size": AUTO_TUNE_BATCH_SIZE,
    "batch_size_candidates": BATCH_SIZE_CANDIDATES,
    "target_effective_batch": TARGET_EFFECTIVE_BATCH,
    "per_device_batch": PER_DEVICE_BATCH,
    "gradient_accumulation_steps": GRAD_ACCUM,
    "effective_batch": PER_DEVICE_BATCH * GRAD_ACCUM,
    "learning_rate": LEARNING_RATE,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "save_steps": SAVE_STEPS,
    "save_total_limit": SAVE_TOTAL_LIMIT,
    "resume_from_checkpoint": resume_checkpoint,
    "prompt_configs": prompt_configs,
}
(FINAL_DIR / "rukopys_oof_train_config.json").write_text(
    json.dumps(train_config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Saved final LoRA adapter to", FINAL_DIR)
print("Trainer checkpoints are in", TRAINER_OUTPUT_DIR)


In [ ]:
# Optional quick sanity check on one cached crop. This is not a leaderboard estimate.
RUN_QUICK_SANITY = True

if RUN_QUICK_SANITY and samples:
    model.eval()
    sample = samples[0]
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": sample["image_path"], "max_pixels": MAX_PIXELS_CROP},
                {"type": "text", "text": sample["prompt"]},
            ],
        }
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt")
    inputs = inputs.to(model.device)
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=TORCH_DTYPE):
        out = model.generate(**inputs, max_new_tokens=192, do_sample=False)
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out)]
    pred = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    print("Prediction:", pred[:1000])
    print("Target:", sample.get("answer", "")[:1000])
